# Amazon Product Review Sentiment Analysis (10-Fold Cross Validation)

**Deployment Link:** [https://amazon-sentiment-hassan.streamlit.app](https://amazon-sentiment-hassan.streamlit.app)

**Name:** Muhammad Hassan Yousaf  
**Roll Number:** FA24-BSE-082  
**Course:** Machine Learning Fundamentals (AIC354), Assignment 4, Task 2  
**Instructor:** Dr. Rao Muhammad Adeel Nawab  
**University:** COMSATS University Islamabad, Lahore Campus  
**GitHub:** [https://github.com/MHassan0000/ML-Prediction-Models](https://github.com/MHassan0000/ML-Prediction-Models)

Customer reviews shape what millions of people buy on Amazon every day. The goal here is to build a classifier that reads a product review and decides whether the customer was satisfied or not, based only on the words they wrote. No star rating is used as a feature - just the text.

**Dataset:** Amazon Polarity (HuggingFace) - 25,000 reviews, balanced 50/50 positive and negative  
**Feature Extraction:** TF-IDF Vectorizer (uni-grams and bi-grams, 5000 features)  
**Evaluation:** 10-Fold Cross Validation, Accuracy metric

**6 Algorithms compared:**
1. Logistic Regression
2. Multinomial Naive Bayes
3. Random Forest
4. Linear SVC
5. k-Nearest Neighbors
6. Decision Tree

To run on Google Colab: open this notebook and run all cells from top to bottom. The dataset streams automatically through HuggingFace - no manual upload needed.

## Step 1: Import Libraries

In [ ]:
import os
import re
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import sklearn
import matplotlib
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from tqdm import tqdm

pd.set_option('display.float_format', lambda v: f'{v:.4f}')

print(f'NumPy        : {np.__version__}')
print(f'Pandas       : {pd.__version__}')
print(f'Scikit-learn : {sklearn.__version__}')
print(f'Matplotlib   : {matplotlib.__version__}')

## Step 2: Load Dataset

The Amazon Polarity dataset contains over 3.6 million product reviews from Amazon, labelled positive (4-5 star) or negative (1-2 star). To keep training times reasonable I stream and sample 12,500 reviews from each class, giving a balanced 25,000-review working dataset. The dataset is available on HuggingFace as `fancyzhx/amazon_polarity` and downloads automatically - no Kaggle account or manual file upload needed.

In [ ]:
try:
    from datasets import load_dataset
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'datasets', '-q'], check=True)
    from datasets import load_dataset

N_PER_CLASS = 12500

print(f'Streaming Amazon Polarity dataset ({N_PER_CLASS:,} reviews per class)...')
ds = load_dataset('fancyzhx/amazon_polarity', split='train', streaming=True)

positives_text, negatives_text = [], []
positives_lbl,  negatives_lbl  = [], []

for item in ds:
    text = ((item.get('title') or '') + ' ' + (item.get('content') or '')).strip()
    if not text:
        continue
    if item['label'] == 1 and len(positives_text) < N_PER_CLASS:
        positives_text.append(text); positives_lbl.append(1)
    elif item['label'] == 0 and len(negatives_text) < N_PER_CLASS:
        negatives_text.append(text); negatives_lbl.append(0)
    total = len(positives_text) + len(negatives_text)
    if total % 5000 == 0 and total > 0:
        print(f'  Collected: {len(positives_text):,} positive, {len(negatives_text):,} negative')
    if len(positives_text) >= N_PER_CLASS and len(negatives_text) >= N_PER_CLASS:
        break

# Build with explicit types so there is no PyArrow backing
all_texts = positives_text + negatives_text
all_labels = positives_lbl + negatives_lbl
all_sentiments = ['positive'] * len(positives_text) + ['negative'] * len(negatives_text)
df = pd.DataFrame({'text': all_texts, 'label': all_labels, 'sentiment': all_sentiments})
df['label'] = df['label'].astype(int)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f'\nDataset ready: {len(df):,} reviews')
print(f'Shape: {df.shape}')
df.head(5)

## Step 3: Understand the Data

In [ ]:
class_counts = df['sentiment'].value_counts()
class_pct    = df['sentiment'].value_counts(normalize=True) * 100

print('Class distribution:')
for label in class_counts.index:
    print(f'  {label:>8}: {class_counts[label]:>6,} reviews ({class_pct[label]:.1f}%)')
print(f'\nTotal reviews : {len(df):,}')
print(f'Missing values: {df.isnull().sum().sum()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2ecc71', '#e74c3c']
bars = axes[0].bar(class_counts.index, class_counts.values, color=colors, edgecolor='white', width=0.5)
for bar, count, pct in zip(bars, class_counts.values, class_pct.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 100,
                 f'{count:,}\n({pct:.1f}%)', ha='center', fontweight='bold')
axes[0].set_title('Review Sentiment Distribution', fontsize=13)
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Number of Reviews')
axes[0].spines[['top', 'right']].set_visible(False)

df['text_length'] = df['text'].apply(len)
for label, color in zip(['positive', 'negative'], colors):
    axes[1].hist(df[df['sentiment'] == label]['text_length'], bins=60,
                 alpha=0.7, label=label, color=color, edgecolor='white')
axes[1].set_title('Review Length Distribution', fontsize=13)
axes[1].set_xlabel('Number of Characters')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
print('Sample POSITIVE reviews:')
for _, row in df[df['sentiment'] == 'positive'].head(3).iterrows():
    print(f'  "{row["text"][:120]}"')
    print()

print('Sample NEGATIVE reviews:')
for _, row in df[df['sentiment'] == 'negative'].head(3).iterrows():
    print(f'  "{row["text"][:120]}"')
    print()

print(f'Average review length: {df["text_length"].mean():.0f} characters')
print(f'Longest review       : {df["text_length"].max()} characters')
print(f'Shortest review      : {df["text_length"].min()} characters')

## Step 4: Pre-processing

Each review goes through the same cleaning steps: lowercase conversion, URL removal, digit stripping, and punctuation removal. Amazon reviews are mostly well-formed English so the main job is normalizing case and stripping noise so TF-IDF can focus on actual words.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print('Before and after cleaning:')
for idx in [0, 2, 5]:
    orig    = df.loc[idx, 'text']
    cleaned = clean_text(orig)
    label   = df.loc[idx, 'sentiment'].upper()
    print(f'\n  [{label}]')
    print(f'  BEFORE: "{orig[:100]}"')
    print(f'  AFTER : "{cleaned[:100]}"')

df['clean_text'] = df['text'].apply(clean_text)
print(f'\nCleaned {len(df):,} reviews.')

## Step 5: Feature Extraction

TF-IDF converts each review into a numerical vector. Each dimension corresponds to a word or two-word phrase (bigram), and its value reflects how distinctive that term is for a particular review. Using bigrams alongside unigrams lets the model pick up phrases like 'not good' or 'highly recommend' which carry clearer sentiment than either word alone.

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

X = tfidf.fit_transform(df['clean_text'])
y = df['label'].values

print(f'TF-IDF matrix shape : {X.shape}')
print(f'Vocabulary size     : {len(tfidf.vocabulary_):,} features')
print(f'Non-zero entries    : {X.nnz:,}')
print(f'Sparsity            : {(1 - X.nnz / (X.shape[0] * X.shape[1])) * 100:.2f}%')

In [ ]:
feature_names = tfidf.get_feature_names_out()
mean_scores   = X.mean(axis=0).A1
top_indices   = mean_scores.argsort()[::-1][:20]

print('Top 20 features by mean TF-IDF score:')
for rank, idx in enumerate(top_indices, 1):
    print(f'  {rank:>2}. {feature_names[idx]:<20} {mean_scores[idx]:.6f}')

## Step 6: Define 6 Algorithms

In [ ]:
algorithms = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Multinomial NB':      MultinomialNB(),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Linear SVC':          LinearSVC(max_iter=2000, random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
}

print(f'{len(algorithms)} algorithms ready:')
for i, name in enumerate(algorithms, 1):
    print(f'  {i}. {name}')

## Step 7: Set Up 10-Fold Cross Validation

In [ ]:
K  = 10
kf = KFold(n_splits=K, shuffle=True, random_state=42)

print(f'K = {K} folds | Total reviews: {X.shape[0]:,}')
print(f'Training per fold : ~{X.shape[0] - X.shape[0] // K:,}')
print(f'Test per fold     : ~{X.shape[0] // K:,}')

print(f'\n  {"Fold":<6} {"Train":>8} {"Test":>8}')
for i, (tr, te) in enumerate(kf.split(X), 1):
    print(f'  {i:<6} {len(tr):>8,} {len(te):>8,}')

## Step 8: Run 10-Fold Cross Validation

The TF-IDF vectorizer is fitted on the training portion of each fold only, then applied to the test portion. This prevents any test vocabulary from leaking into the feature representation. Each model is cloned and trained fresh for every fold.

In [ ]:
results          = {}
first_fold_preds = {}
total_tasks      = len(algorithms) * K

print(f'Running {K}-fold CV for {len(algorithms)} algorithms ({total_tasks} total fits)...')

X_texts = df['clean_text'].tolist()
y       = df['label'].astype(int).values
with tqdm(total=total_tasks, desc='Cross Validation', unit='fit') as pbar:
    for algo_name, algo in algorithms.items():
        fold_accs = []
        for fold_i, (tr, te) in enumerate(kf.split(X_texts), 1):
            vec  = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
            X_tr = vec.fit_transform(X_texts[tr])
            X_te = vec.transform(X_texts[te])

            m = clone(algo)
            m.fit(X_tr, y[tr])
            y_pred = m.predict(X_te)
            fold_accs.append(accuracy_score(y[te], y_pred))

            if fold_i == 1:
                first_fold_preds[algo_name] = (y[te], y_pred)
            pbar.update(1)

        results[algo_name] = fold_accs
        print(f'  {algo_name:<22} -> {np.mean(fold_accs)*100:.2f}% +/- {np.std(fold_accs)*100:.2f}%')

print('\nDone!')

## Step 9: Compare Results

### Per-Fold Accuracy

In [ ]:
fold_df = pd.DataFrame(results, index=[f'Fold {i}' for i in range(1, K + 1)]).T
print('Per-fold accuracy for each algorithm:')
(fold_df * 100).round(2)

### Summary Table

In [ ]:
summary = pd.DataFrame({
    'Mean Accuracy (%)': [np.mean(v) * 100 for v in results.values()],
    'Std Dev (%)':       [np.std(v)  * 100 for v in results.values()],
    'Min Accuracy (%)':  [np.min(v)  * 100 for v in results.values()],
    'Max Accuracy (%)':  [np.max(v)  * 100 for v in results.values()],
}, index=results.keys()).sort_values('Mean Accuracy (%)', ascending=False)

summary.insert(0, 'Rank', range(1, len(summary) + 1))
print('Algorithm comparison sorted by mean accuracy:')
summary

### Charts

In [ ]:
COLORS       = ['#2ecc71', '#3498db', '#9b59b6', '#e67e22', '#e74c3c', '#1abc9c']
sorted_algos = summary.index.tolist()
sorted_means = [np.mean(results[a]) * 100 for a in sorted_algos]
sorted_stds  = [np.std(results[a])  * 100 for a in sorted_algos]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

bars = axes[0].barh(sorted_algos, sorted_means, xerr=sorted_stds,
                    color=COLORS, edgecolor='white', capsize=4, height=0.6)
axes[0].set_yticks(range(len(sorted_algos)))
axes[0].set_yticklabels(sorted_algos)
axes[0].set_xlabel('Accuracy (%)')
axes[0].set_title('Mean Accuracy with Std Dev', fontsize=13)
axes[0].invert_yaxis()
axes[0].spines[['top', 'right']].set_visible(False)
for i, (mean, std) in enumerate(zip(sorted_means, sorted_stds)):
    axes[0].text(mean + std + 0.2, i, f'{mean:.2f}%', va='center', fontsize=9, fontweight='bold')

box_data = [np.array(results[a]) * 100 for a in sorted_algos]
bp = axes[1].boxplot(box_data, vert=True, patch_artist=True, labels=sorted_algos)
for patch, c in zip(bp['boxes'], COLORS):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy Distribution per Algorithm', fontsize=13)
axes[1].tick_params(axis='x', rotation=45)
axes[1].spines[['top', 'right']].set_visible(False)

for i, algo in enumerate(sorted_algos):
    axes[2].plot(range(1, K + 1), np.array(results[algo]) * 100, marker='o',
                 label=algo, color=COLORS[i], linewidth=2)
axes[2].set_xlabel('Fold')
axes[2].set_ylabel('Accuracy (%)')
axes[2].set_title('Accuracy Across All 10 Folds', fontsize=13)
axes[2].legend(fontsize=8)
axes[2].set_xticks(range(1, K + 1))
axes[2].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

## Step 10: Observations

In [ ]:
mean_accs  = {n: np.mean(v) * 100 for n, v in results.items()}
best_algo  = max(mean_accs, key=mean_accs.get)
worst_algo = min(mean_accs, key=mean_accs.get)

print(f'Best  algorithm : {best_algo}')
print(f'  Mean Accuracy : {mean_accs[best_algo]:.2f}%')
print(f'  Std Deviation : {np.std(results[best_algo]) * 100:.2f}%')
print()
print(f'Worst algorithm : {worst_algo}')
print(f'  Mean Accuracy : {mean_accs[worst_algo]:.2f}%')
print()
print(f'Accuracy gap    : {mean_accs[best_algo] - mean_accs[worst_algo]:.2f}%')
print()

y_test_b, y_pred_b = first_fold_preds[best_algo]
print(f'Classification Report for {best_algo} (Fold 1):')
print(classification_report(y_test_b, y_pred_b, target_names=['negative', 'positive'], digits=4))

## Step 11: Train Final Model

The best algorithm is now trained on the full 25,000-review dataset. The model and vectorizer are saved as pickle files for use in the web application.

In [ ]:
print(f'Training final model: {best_algo}')

final_tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_all = final_tfidf.fit_transform(df['clean_text'])

final_model = clone(algorithms[best_algo])
final_model.fit(X_all, y)

print(f'Trained on {X_all.shape[0]:,} reviews with {X_all.shape[1]:,} features.')

with open('amazon_best_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)

with open('amazon_vectorizer.pkl', 'wb') as f:
    pickle.dump({'vectorizer': final_tfidf}, f)

print(f'Saved amazon_best_model.pkl  ({os.path.getsize("amazon_best_model.pkl") / 1024:.1f} KB)')
print(f'Saved amazon_vectorizer.pkl  ({os.path.getsize("amazon_vectorizer.pkl") / 1024:.1f} KB)')

## Step 12: Test Predictions

In [ ]:
with open('amazon_best_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)
with open('amazon_vectorizer.pkl', 'rb') as f:
    loaded_vec = pickle.load(f)['vectorizer']

print('Model loaded successfully!')

test_reviews = [
    'This product is absolutely amazing. Best purchase I have ever made. Highly recommend!',
    'Complete waste of money. Broke after two days. Terrible quality, do not buy.',
    'Exactly as described, fast shipping and great quality for the price.',
    'Very disappointed. The color looked nothing like the pictures. Returning it.',
    'My kids love this toy. Kept them busy for hours. Great gift idea.',
    'Poor customer service and the item arrived damaged. Very frustrating.',
]

label_map = {0: 'NEGATIVE', 1: 'POSITIVE'}
icons     = {0: '---', 1: '+++'}

print(f'\nTest predictions ({best_algo}):')
print('=' * 70)
for review in test_reviews:
    cleaned = clean_text(review)
    pred    = loaded_model.predict(loaded_vec.transform([cleaned]))[0]
    label   = label_map[pred]
    icon    = icons[pred]
    print(f'  [{icon}]  {label}  -  "{review[:60]}..."')

## Step 13: Ready for Deployment

In [ ]:
print('Amazon Review Sentiment Analysis - Project Summary')
print('=' * 55)
print(f'  Problem         : Amazon Product Review Sentiment (Binary Text Classification)')
print(f'  Dataset         : Amazon Polarity - {len(df):,} reviews (balanced)')
print(f'  Feature method  : TF-IDF Vectorizer (max_features=5000, ngrams=(1,2))')
print(f'  Evaluation      : 10-Fold Cross Validation')
print(f'  Best algorithm  : {best_algo}')
print(f'  CV Accuracy     : {mean_accs[best_algo]:.2f}%')
print(f'  Saved model     : amazon_best_model.pkl')
print(f'  Saved vectorizer: amazon_vectorizer.pkl')
print()
print(f'  Deployment link : https://amazon-sentiment-hassan.streamlit.app')
print(f'  GitHub          : https://github.com/MHassan0000/ML-Prediction-Models')